# 🛒 E-Commerce Supply Chain Intelligence

**Project:** End-to-end supply chain analysis for a beauty & personal care e-commerce brand  
**Dataset:** 100 SKUs across Skincare, Haircare & Cosmetics categories  
**Objective:** Uncover revenue trends, shipping efficiency, supplier performance, defect patterns, and logistics cost drivers.

---
### Business Questions
1. How does product price relate to revenue generated?
2. Which product category drives the most sales volume?
3. Which shipping carrier generates the most revenue?
4. How do shipping costs compare across carriers?
5. Which product type has the highest defect rate?
6. How are costs distributed across transportation modes?
7. How do manufacturing costs vary by supplier?
8. What does the stock vs lead time picture look like?

---
### Notebook Sections
1. Library Imports  
2. Data Loading & EDA  
3. Revenue vs Price Analysis  
4. Sales Volume by Product Category  
5. Revenue & Cost by Shipping Carrier  
6. SKU-Level Revenue & Order Analysis  
7. Transportation Mode Cost Distribution  
8. Defect Rate Analysis  
9. Supplier & Location Insights  
10. Summary Dashboard

## 1. Library Imports

In [ ]:
# ── Core Libraries ───────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Visualisation ────────────────────────────────────────────────
import plotly.express       as px
import plotly.graph_objects as go
import plotly.io            as pio
from plotly.subplots import make_subplots

# ── Theme ────────────────────────────────────────────────────────
pio.templates.default = 'plotly_white'

# ── Colour Palette (consistent across all charts) ───────────────
PALETTE_CATEGORY  = ['#6C63FF', '#FF6584', '#43D9AD']          # per product type
PALETTE_CARRIER   = ['#F4A261', '#E76F51', '#2A9D8F']          # per carrier
PALETTE_TRANSPORT = ['#264653', '#E9C46A', '#E76F51', '#2A9D8F']

print('✅ All libraries imported successfully.')

## 2. Data Loading & Exploratory Analysis

In [ ]:
# Load the product supply dataset
supply_df = pd.read_csv('../data/product_supply_data.csv')

print(f'Dataset shape : {supply_df.shape[0]} rows × {supply_df.shape[1]} columns')
print(f'Product types : {supply_df["Product type"].unique()}')
print(f'Carriers      : {supply_df["Shipping carriers"].unique()}')
print(f'Locations     : {supply_df["Location"].unique()}')
supply_df.head(10)

In [ ]:
# Statistical summary of all numeric columns
supply_df.describe().round(2)

In [ ]:
# ── Data Quality Check ───────────────────────────────────────────
print('── Missing Values ─────────────────────────────────')
missing = supply_df.isnull().sum()
print(missing[missing > 0] if missing.any() else '✅ No missing values found')

print('\n── Duplicate Rows ──────────────────────────────────')
dupes = supply_df.duplicated().sum()
print(f'Duplicates found: {dupes}')

print('\n── Inspection Result Breakdown ─────────────────────')
print(supply_df['Inspection results'].value_counts())

print('\n── Customer Demographics Breakdown ─────────────────')
print(supply_df['Customer demographics'].value_counts())

In [ ]:
# ── Engineered Columns ───────────────────────────────────────────
supply_df['profit_estimate']      = supply_df['Revenue generated'] - supply_df['Manufacturing costs'] - supply_df['Shipping costs']
supply_df['revenue_per_unit']     = supply_df['Revenue generated'] / supply_df['Number of products sold']
supply_df['cost_efficiency']      = supply_df['Revenue generated'] / supply_df['Costs']
supply_df['shipping_cost_per_unit']= supply_df['Shipping costs'] / supply_df['Order quantities']

print('✅ Feature engineering complete.')
print(f'   New columns: profit_estimate, revenue_per_unit, cost_efficiency, shipping_cost_per_unit')

## 3. Revenue vs Price — Scatter with Trend Line

In [ ]:
# Scatter: Price vs Revenue Generated coloured by product type
price_revenue_fig = px.scatter(
    supply_df,
    x                = 'Price',
    y                = 'Revenue generated',
    color            = 'Product type',
    size             = 'Number of products sold',
    hover_data       = ['SKU', 'Number of products sold', 'Manufacturing costs'],
    trendline        = 'ols',
    color_discrete_sequence = PALETTE_CATEGORY,
    title            = '💰 Product Price vs Revenue Generated (size = units sold)',
    labels           = {'Price': 'Unit Price (₹)', 'Revenue generated': 'Total Revenue (₹)'}
)

price_revenue_fig.update_layout(
    title_font_size  = 18,
    legend_title     = 'Product Category',
    plot_bgcolor     = '#FAFAFA',
    hovermode        = 'closest'
)
price_revenue_fig.show()

In [ ]:
# Box plot: Revenue spread per product type
rev_box_fig = px.box(
    supply_df,
    x     = 'Product type',
    y     = 'Revenue generated',
    color = 'Product type',
    color_discrete_sequence = PALETTE_CATEGORY,
    title = '📦 Revenue Distribution by Product Category',
    points= 'all',
    notched=True
)
rev_box_fig.update_layout(showlegend=False)
rev_box_fig.show()

## 4. Sales Volume by Product Category

In [ ]:
# Group: total units sold per product type
category_sales = (
    supply_df
    .groupby('Product type')['Number of products sold']
    .sum()
    .reset_index()
    .rename(columns={'Number of products sold': 'total_units_sold'})
    .sort_values('total_units_sold', ascending=False)
)
print('Sales by Product Category:')
print(category_sales)

In [ ]:
# Donut chart: sales share by category
sales_donut = px.pie(
    category_sales,
    values  = 'total_units_sold',
    names   = 'Product type',
    title   = '🛍️ Units Sold Share by Product Category',
    hole    = 0.55,
    color_discrete_sequence = PALETTE_CATEGORY,
    hover_data = ['total_units_sold']
)
sales_donut.update_traces(
    textposition = 'inside',
    textinfo     = 'percent+label',
    hovertemplate= '<b>%{label}</b><br>Units: %{value:,}<br>Share: %{percent}<extra></extra>'
)
sales_donut.update_layout(
    title_font_size = 18,
    annotations     = [dict(text='Sales<br>Split', x=0.5, y=0.5, font_size=14, showarrow=False)]
)
sales_donut.show()

In [ ]:
# Grouped: avg order quantity & avg revenue per category
category_deep = (
    supply_df
    .groupby('Product type')
    .agg(
        avg_order_qty     = ('Order quantities',          'mean'),
        avg_revenue       = ('Revenue generated',         'mean'),
        avg_price         = ('Price',                     'mean'),
        avg_stock         = ('Stock levels',              'mean'),
        total_revenue     = ('Revenue generated',         'sum'),
        sku_count         = ('SKU',                       'count')
    )
    .round(2)
    .reset_index()
)
print('Deep Category Analysis:')
print(category_deep.to_string(index=False))

In [ ]:
# Horizontal bar: total revenue per category
rev_bar = px.bar(
    category_deep.sort_values('total_revenue'),
    x           = 'total_revenue',
    y           = 'Product type',
    orientation = 'h',
    color       = 'Product type',
    color_discrete_sequence = PALETTE_CATEGORY,
    text        = 'total_revenue',
    title       = '📊 Total Revenue by Product Category'
)
rev_bar.update_traces(texttemplate='₹%{text:,.0f}', textposition='outside')
rev_bar.update_layout(showlegend=False, xaxis_title='Total Revenue (₹)')
rev_bar.show()

## 5. Shipping Carrier Analysis — Revenue & Cost

In [ ]:
# Group: revenue and cost per carrier
carrier_summary = (
    supply_df
    .groupby('Shipping carriers')
    .agg(
        total_revenue   = ('Revenue generated', 'sum'),
        total_ship_cost = ('Shipping costs',    'sum'),
        avg_ship_cost   = ('Shipping costs',    'mean'),
        shipment_count  = ('SKU',               'count')
    )
    .round(2)
    .reset_index()
)
print('Carrier Summary:')
print(carrier_summary.to_string(index=False))

In [ ]:
# Pie: revenue share per carrier
carrier_revenue_pie = px.pie(
    carrier_summary,
    values  = 'total_revenue',
    names   = 'Shipping carriers',
    title   = '🚚 Total Revenue Contribution by Shipping Carrier',
    hole    = 0.5,
    color_discrete_sequence = PALETTE_CARRIER,
)
carrier_revenue_pie.update_traces(
    textposition = 'inside',
    textinfo     = 'percent+label',
    hovertemplate= '<b>%{label}</b><br>Revenue: ₹%{value:,.0f}<br>Share: %{percent}<extra></extra>'
)
carrier_revenue_pie.update_layout(
    title_font_size=18,
    annotations=[dict(text='Revenue<br>Share', x=0.5, y=0.5, font_size=13, showarrow=False)]
)
carrier_revenue_pie.show()

In [ ]:
# Grouped bar: revenue vs shipping cost per carrier
carrier_compare = go.Figure()
carrier_compare.add_trace(go.Bar(
    name = 'Total Revenue',
    x    = carrier_summary['Shipping carriers'],
    y    = carrier_summary['total_revenue'],
    marker_color = PALETTE_CARRIER[0],
    text = carrier_summary['total_revenue'].round(0),
    texttemplate = '₹%{text:,.0f}',
    textposition = 'outside'
))
carrier_compare.add_trace(go.Bar(
    name = 'Shipping Cost',
    x    = carrier_summary['Shipping carriers'],
    y    = carrier_summary['total_ship_cost'],
    marker_color = PALETTE_CARRIER[2],
    text = carrier_summary['total_ship_cost'].round(0),
    texttemplate = '₹%{text:,.0f}',
    textposition = 'outside'
))
carrier_compare.update_layout(
    barmode     = 'group',
    title       = '📦 Carrier Revenue vs Shipping Cost Comparison',
    xaxis_title = 'Shipping Carrier',
    yaxis_title = 'Amount (₹)',
    title_font_size = 18
)
carrier_compare.show()

In [ ]:
# Bar: shipping cost per carrier
ship_cost_fig = px.bar(
    carrier_summary,
    x           = 'Shipping carriers',
    y           = 'total_ship_cost',
    color       = 'Shipping carriers',
    text        = 'total_ship_cost',
    color_discrete_sequence = PALETTE_CARRIER,
    title       = '💸 Total Shipping Cost by Carrier'
)
ship_cost_fig.update_traces(texttemplate='₹%{text:.1f}', textposition='outside')
ship_cost_fig.update_layout(showlegend=False, title_font_size=18)
ship_cost_fig.show()

## 6. SKU-Level Revenue & Order Quantity Trends

In [ ]:
# Sort SKUs by revenue for cleaner line chart
sku_sorted = supply_df.sort_values('Revenue generated', ascending=False).reset_index(drop=True)

# Line chart: revenue per SKU (sorted descending)
sku_rev_fig = px.line(
    sku_sorted,
    x           = 'SKU',
    y           = 'Revenue generated',
    color       = 'Product type',
    markers     = True,
    color_discrete_sequence = PALETTE_CATEGORY,
    title       = '📈 Revenue Generated per SKU (sorted by Revenue)',
    labels      = {'Revenue generated': 'Revenue (₹)', 'SKU': 'Stock Keeping Unit'}
)
sku_rev_fig.update_layout(title_font_size=18, xaxis_tickangle=-45)
sku_rev_fig.show()

In [ ]:
# Line chart: order quantity per SKU
sku_order_fig = px.line(
    supply_df.sort_values('Order quantities', ascending=False),
    x           = 'SKU',
    y           = 'Order quantities',
    color       = 'Product type',
    markers     = True,
    color_discrete_sequence = PALETTE_CATEGORY,
    title       = '📦 Order Quantity per SKU (sorted by Quantity)',
    labels      = {'Order quantities': 'Qty Ordered', 'SKU': 'Stock Keeping Unit'}
)
sku_order_fig.update_layout(title_font_size=18, xaxis_tickangle=-45)
sku_order_fig.show()

In [ ]:
# Scatter: Stock Levels vs Lead Times (inventory pressure view)
stock_lead_fig = px.scatter(
    supply_df,
    x           = 'Lead times',
    y           = 'Stock levels',
    color       = 'Product type',
    size        = 'Order quantities',
    hover_data  = ['SKU', 'Supplier name', 'Location'],
    color_discrete_sequence = PALETTE_CATEGORY,
    title       = '🏭 Stock Levels vs Lead Times (size = order qty)'
)
stock_lead_fig.update_layout(title_font_size=18)
stock_lead_fig.show()

## 7. Transportation Mode — Cost Distribution

In [ ]:
# Group: total logistics cost per transport mode
transport_cost = (
    supply_df
    .groupby('Transportation modes')
    .agg(
        total_logistics_cost = ('Costs',              'sum'),
        avg_logistics_cost   = ('Costs',              'mean'),
        shipment_count       = ('SKU',                'count'),
        total_revenue        = ('Revenue generated',  'sum')
    )
    .round(2)
    .reset_index()
)
print('Transport Mode Cost Summary:')
print(transport_cost.to_string(index=False))

In [ ]:
# Styled donut: logistics cost share by transport mode
transport_donut = go.Figure(data=[go.Pie(
    labels     = transport_cost['Transportation modes'],
    values     = transport_cost['total_logistics_cost'],
    hole       = 0.45,
    hoverinfo  = 'label+value+percent',
    textinfo   = 'percent+label',
    textfont   = dict(size=14),
    marker     = dict(
        colors = PALETTE_TRANSPORT,
        line   = dict(color='white', width=2)
    )
)])
transport_donut.update_layout(
    title      = '🚛 Logistics Cost Distribution by Transportation Mode',
    title_font_size = 18,
    annotations= [dict(text='Mode<br>Cost', x=0.5, y=0.5, font_size=14, showarrow=False)]
)
transport_donut.show()

In [ ]:
# Bar: Revenue vs Logistics Cost per transport mode
transport_compare = px.bar(
    transport_cost,
    x           = 'Transportation modes',
    y           = ['total_revenue', 'total_logistics_cost'],
    barmode     = 'group',
    color_discrete_sequence = [PALETTE_CARRIER[0], PALETTE_CARRIER[2]],
    title       = '⚖️ Revenue vs Logistics Cost by Transportation Mode',
    labels      = {'value': 'Amount (₹)', 'variable': 'Metric'}
)
transport_compare.update_layout(title_font_size=18)
transport_compare.show()

## 8. Defect Rate Analysis

In [ ]:
# Average defect rate per product type
defect_by_category = (
    supply_df
    .groupby('Product type')['Defect rates']
    .agg(['mean', 'max', 'min', 'std'])
    .round(4)
    .reset_index()
    .rename(columns={'mean':'avg_defect','max':'max_defect','min':'min_defect','std':'std_defect'})
)
print('Defect Rate by Product Category:')
print(defect_by_category.to_string(index=False))

In [ ]:
# Bar: average defect rate per category
defect_bar = px.bar(
    defect_by_category,
    x           = 'Product type',
    y           = 'avg_defect',
    color       = 'Product type',
    error_y     = 'std_defect',
    color_discrete_sequence = PALETTE_CATEGORY,
    text        = 'avg_defect',
    title       = '🔍 Average Defect Rate by Product Category (with std dev bars)'
)
defect_bar.update_traces(texttemplate='%{text:.4f}', textposition='outside')
defect_bar.update_layout(showlegend=False, title_font_size=18,
                         yaxis_title='Average Defect Rate')
defect_bar.show()

In [ ]:
# Defect rate vs inspection result breakdown
inspection_defect = (
    supply_df
    .groupby(['Product type', 'Inspection results'])['Defect rates']
    .mean()
    .round(4)
    .reset_index()
)

defect_inspect_fig = px.bar(
    inspection_defect,
    x           = 'Product type',
    y           = 'Defect rates',
    color       = 'Inspection results',
    barmode     = 'group',
    title       = '🔬 Defect Rate by Category & Inspection Result',
    color_discrete_sequence = ['#E76F51', '#2A9D8F', '#E9C46A']
)
defect_inspect_fig.update_layout(title_font_size=18)
defect_inspect_fig.show()

## 9. Supplier & Location Insights

In [ ]:
# Supplier: manufacturing cost & production volume
supplier_summary = (
    supply_df
    .groupby('Supplier name')
    .agg(
        avg_mfg_cost       = ('Manufacturing costs', 'mean'),
        total_production   = ('Production volumes',  'sum'),
        avg_lead_time      = ('Manufacturing lead time', 'mean'),
        avg_defect_rate    = ('Defect rates',         'mean'),
        sku_count          = ('SKU',                  'count')
    )
    .round(2)
    .reset_index()
)
print('Supplier Performance Overview:')
print(supplier_summary.to_string(index=False))

In [ ]:
# Bubble chart: Supplier — Mfg Cost vs Production Volume vs Defect Rate
supplier_bubble = px.scatter(
    supplier_summary,
    x           = 'avg_mfg_cost',
    y           = 'total_production',
    size        = 'avg_defect_rate',
    color       = 'Supplier name',
    text        = 'Supplier name',
    title       = '🏭 Supplier: Mfg Cost vs Production (bubble = defect rate)',
    labels      = {'avg_mfg_cost':'Avg Mfg Cost (₹)', 'total_production':'Total Production Volume'}
)
supplier_bubble.update_traces(textposition='top center')
supplier_bubble.update_layout(title_font_size=18)
supplier_bubble.show()

In [ ]:
# Revenue by city/location
location_rev = (
    supply_df
    .groupby('Location')
    .agg(
        total_revenue  = ('Revenue generated', 'sum'),
        avg_revenue    = ('Revenue generated', 'mean'),
        sku_count      = ('SKU',               'count')
    )
    .round(2)
    .reset_index()
    .sort_values('total_revenue', ascending=False)
)

location_fig = px.bar(
    location_rev,
    x           = 'Location',
    y           = 'total_revenue',
    color       = 'sku_count',
    text        = 'total_revenue',
    color_continuous_scale = 'Viridis',
    title       = '🏙️ Total Revenue by City Location (colour = SKU count)'
)
location_fig.update_traces(texttemplate='₹%{text:,.0f}', textposition='outside')
location_fig.update_layout(title_font_size=18, coloraxis_showscale=True)
location_fig.show()

## 10. Summary — Key Metrics Snapshot

In [ ]:
print('=' * 60)
print('   E-COMMERCE SUPPLY CHAIN — SUMMARY STATISTICS')
print('=' * 60)
print(f'  Total SKUs Analysed         : {len(supply_df):,}')
print(f'  Total Revenue (₹)           : {supply_df["Revenue generated"].sum():>14,.2f}')
print(f'  Avg Revenue per SKU (₹)     : {supply_df["Revenue generated"].mean():>14,.2f}')
print(f'  Total Units Sold            : {supply_df["Number of products sold"].sum():>14,}')
print(f'  Total Shipping Cost (₹)     : {supply_df["Shipping costs"].sum():>14,.2f}')
print(f'  Total Logistics Cost (₹)    : {supply_df["Costs"].sum():>14,.2f}')
print(f'  Avg Defect Rate             : {supply_df["Defect rates"].mean():>14.4f}')
top_cat   = supply_df.groupby("Product type")["Number of products sold"].sum().idxmax()
top_carry = supply_df.groupby("Shipping carriers")["Revenue generated"].sum().idxmax()
top_loc   = supply_df.groupby("Location")["Revenue generated"].sum().idxmax()
print(f'  Top-Selling Category        : {top_cat}')
print(f'  Top Revenue Carrier         : {top_carry}')
print(f'  Top Revenue Location        : {top_loc}')
print('=' * 60)

In [ ]:
# 2x2 Summary subplot grid
fig_summary = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Revenue by Category', 'Shipping Cost by Carrier',
        'Avg Defect Rate', 'Avg Manufacturing Cost by Supplier'
    ),
    specs=[[{'type':'bar'},{'type':'bar'}],[{'type':'bar'},{'type':'bar'}]]
)

# Revenue by category
fig_summary.add_trace(go.Bar(
    x=category_deep['Product type'], y=category_deep['total_revenue'],
    marker_color=PALETTE_CATEGORY, name='Revenue', showlegend=False
), row=1, col=1)

# Shipping cost by carrier
fig_summary.add_trace(go.Bar(
    x=carrier_summary['Shipping carriers'], y=carrier_summary['total_ship_cost'],
    marker_color=PALETTE_CARRIER, name='Ship Cost', showlegend=False
), row=1, col=2)

# Avg defect rate
fig_summary.add_trace(go.Bar(
    x=defect_by_category['Product type'], y=defect_by_category['avg_defect'],
    marker_color=PALETTE_CATEGORY, name='Defect', showlegend=False
), row=2, col=1)

# Avg mfg cost by supplier
fig_summary.add_trace(go.Bar(
    x=supplier_summary['Supplier name'], y=supplier_summary['avg_mfg_cost'],
    marker_color='#6C63FF', name='Mfg Cost', showlegend=False
), row=2, col=2)

fig_summary.update_layout(
    height=700,
    title_text='📋 E-Commerce Supply Chain — Summary Dashboard',
    title_font_size=18,
    plot_bgcolor='#FAFAFA'
)
fig_summary.show()